<a href="https://colab.research.google.com/github/Squad-Nina-da-Hora/wmc-desafio-previsao-demencia/blob/steph%2Ffeature%2Fanalise/feature/analise/analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Squad-Nina-da-Hora/wmc-desafio-previsao-demencia/blob/main/analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# **Análise de Risco de Alzheimer**
---


🎯 **Objetivo:**  Prever sinais de demência, através de informações clínicas e demográficas de pacientes com potencial risco de Alzheimer (OASIS).


---


Desafio Estatística com Python - Classificação

Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## 1. Configurações Iniciais

Variáveis da base de dados:

- `Age`: Idade do paciente (numérico)
- `Sex`: Gênero (F: feminino, M: masculino)
- `EDUC`: Anos de escolaridade (numérico)
- `SES`: Status socioeconômico (1 a 5)
- `MMSE`: Escore do Mini Exame do Estado Mental (0 a 30)
- `CDR`: Clinical Dementia Rating (0 a 3)
- `eTIV`: Volume intracraniano estimado
- `nWBV`: Proporção de volume cerebral normalizado
- `ASF`: Fator de escala anatômica
- `Group` (alvo): Classificação do paciente
  - `Nondemented` - será tratada para variável binária 0
  - `Demented` e `Converted` - serão tratadas para variável binária 1

In [ ]:
# ==============================
# IMPORTACOES
# ==============================

import kagglehub    # Para baixar datasets do Kaggle
import numpy as np  # Para operacoes numericas e arrays
import pandas as pd  # Para manipulaco e analise de data frames
from IPython.display import display, Markdown

# Bibliotecas para criacao de graficos
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno

# Bibliotecas para criacao de modelos de ML
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

# Carregamento da base de dados
arquivo = 'oasis_longitudinal'
url = f'{kagglehub.dataset_download("jboysen/mri-and-alzheimers")}/{arquivo}.csv'
df = pd.read_csv(url).drop(columns=['Subject ID', 'MRI ID', 'Visit', 'MR Delay', 'Hand'])

In [ ]:
# ==============================
# TRATAMENTO INICIAL
# ==============================

# 1. Renomeia algumas colunas
df.rename(columns={
  'Group': 'Demented',
  'M/F': 'Sex'
}, inplace=True)
# 2. Converte a coluna 'Demented' para bool
df['Demented'] = df['Demented'] != 'Nondemented'
# 3. Converte as colunas especificas para categorias
vars_categoricas = ['Sex', 'SES']
for cat in vars_categoricas:
  df[cat] = df[cat].astype('category')

df.head()

In [ ]:
# ==============================
# VARIAVEIS PARA REUTILIZACAO
# ==============================

alvo = 'Demented'

df_numericas_cols = df.select_dtypes(include=['number']).columns
df_categoricas_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns

rotulos_vars = {
  'Demented': 'Status de Demência',
  'Sex': 'Gênero',
  'SES': 'Nível socioeconômico',
  'Age': 'Idade',
  'EDUC': 'Educação',
  'MMSE': 'Pontuação do Mini Exame Mental',
  'CDR': 'Classificação clínica de Demência',
  'eTIV': 'Volume intracraniano estimado',
  'nWBV': 'Volume cerebral normalizado',
  'ASF': 'Fator de escala anatômica'
}

alvo_rotulos = ['Sem Demência', 'Com Demência']

# Configuracos visuais dos graficos
paleta = 'flare'
cores = sns.color_palette(paleta, n_colors=2)

In [ ]:
# ==============================
# FUNCOES PARA REUTILIZACAO
# ==============================

def obter_moda(x):
  """
  Obtem o primeiro valor registrado da moda de uma coluna, caso haja mais
  de um valor com a mesma frequencia.
  """
  return x.mode().iloc[0]


def aplicar_cores(coluna, paleta=paleta):
  """
  Cria uma paleta de cores baseada na quantidade de itens unicos em uma coluna.

  - coluna: A coluna do DataFrame (ex: data['Group'])
  - paleta: O nome do estilo de cores (ex: 'flare')
  """
  return sns.color_palette(paleta, n_colors=coluna.nunique())


def plotar_heatmap(dados, tamanho=[7,6], titulo=None, rotulo_x=None, rotulo_y=None, **kwargs):
  """
  Gera um grafico de calor (heatmap) estilizado utilizando Seaborn.

  Parametros:
    dados (DataFrame/Series): Matriz de dados ou correlacao a ser plotada.
    tamanho (list): Dimensoes do grafico no formato [largura, altura].
    titulo (str): Titulo que sera exibido no topo do grafico.
    rotulo_x/y (str): Rotulo do eixo X/Y.
    **kwargs: Argumentos adicionais passados diretamente para a funcao sns.heatmap.
  """
  plt.figure(figsize=tamanho)
  plt.title(titulo, fontsize=12, fontweight='bold')
  # Define valores padrao, mas permite que o kwargs os sobrescreva
  params = {'annot': True, 'cmap': paleta, 'fmt': '.2f', 'linewidths': 0.5}
  params.update(kwargs)
  sns.heatmap(dados, **params)
  plt.xlabel(rotulo_x)
  plt.ylabel(rotulo_y)
  plt.tight_layout()
  plt.show()


def tratar_nulos(dados_treino, dados_teste=None, tendencia='media'):
  """
  Trata valores ausentes usando as medida de tendencia, calculada apenas no
  treinoe aplicada em treino e teste (evita vazamento de dados).

  Parametros:
    dados_treino (pd.DataFrame): conjunto de treino.
    dados_teste (pd.DataFrame, opcional): conjunto de teste.
    tendencia (str, opcional): 'media', 'mediana' ou 'moda'. Padrao: 'media'.

  Retorno: tupla (X_train, X_test) com nulos preenchidos.
  """
  colunas_com_nulo = dados_treino.columns[dados_treino.isna().any()].tolist()

  for col in colunas_com_nulo:
    if tendencia == 'mediana':
      valor_tendencia = dados_treino[col].median()
    elif tendencia == 'moda':
      valor_tendencia = dados_treino[col].mode()[0]
    else:
      valor_tendencia = dados_treino[col].mean()

    dados_treino[col] = dados_treino[col].fillna(valor_tendencia)

    if dados_teste is not None:
      dados_teste[col] = dados_teste[col].fillna(valor_tendencia)

  return dados_treino, dados_teste

## 2. Análise Exploratória


In [ ]:
df.describe()

In [ ]:
df.describe(include=['object', 'category', 'bool'])

In [ ]:
contagem_nulos = df.isnull().sum()

datadict = pd.DataFrame(df.dtypes)
datadict.columns = ['tipos']
datadict['nulos'] = contagem_nulos
datadict['%_nulos'] = ((contagem_nulos / df.shape[0]) * 100).round(2)
datadict['unicos'] = df.nunique()
datadict

In [ ]:
# Grafico de nulos
msno.matrix(df, figsize=(10, 5), fontsize=9, color=sns.color_palette(paleta)[0])
plt.title('Visualização de Valores Ausentes', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Grafico de distribuicao das variaveis categoricas e alvo

num_plots = len(df_categoricas_cols)

fig, axes = plt.subplots(nrows=1, ncols=num_plots, figsize=(6 * num_plots, 6))

# Garante que 'axes' seja um array mesmo que haja apenas um subplot
if num_plots == 1:
  axes = [axes]

for i, col in enumerate(df_categoricas_cols):
  ax = sns.countplot(x=df[col], stat='percent', ax=axes[i], palette=aplicar_cores(df[col]), hue=df[col], legend=False)

  axes[i].set_xlabel('')
  axes[i].set_ylabel('Proporção de Pacientes (%)' if i == 0 else '') # Apenas o primeiro gráfico tera o rotulo Y

  axes[i].set_title(f'Distribuição de {rotulos_vars.get(col, col)}', fontsize=14, fontweight='bold')

  # Adiciona as porcentagens em cima das barras
  for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%')

plt.tight_layout() # Ajusta o layout para evitar sobreposicao
plt.show()

In [ ]:
# Boxplots das variaveis numericas vs. alvo

num_plots = len(df_numericas_cols)
ncols_3 = 3 # Define o numero de colunas desejado
nrows = (num_plots + ncols_3 - 1) // ncols_3 # Calcula o numero de linhas necessarias

fig, axes = plt.subplots(nrows=nrows, ncols=ncols_3, figsize=(6 * ncols_3, 6 * nrows))

# Achata o array 'axes' para facilitar a iteracao
axes = axes.flatten()

for i, col in enumerate(df_numericas_cols):
  ax = sns.boxplot(data=df, x=alvo, y=col, ax=axes[i], palette=paleta, hue=alvo, legend=False)

  axes[i].set_xticks([0, 1])
  axes[i].set_xticklabels(alvo_rotulos)
  axes[i].set_xlabel(f'{rotulos_vars.get(alvo, alvo)}')
  axes[i].set_title(f'Boxplot de {rotulos_vars.get(col, col)}', fontsize=14, fontweight='bold')

# Remove subplots vazios, se houver
for j in range(num_plots, nrows * ncols_3):
  fig.delaxes(axes[j])

plt.tight_layout() # Ajusta o layout para evitar sobreposicao
plt.show()

In [ ]:
# Histogramas das variaveis numericas vs. alvo

fig, axes = plt.subplots(nrows=nrows, ncols=ncols_3, figsize=(6 * ncols_3, 6 * nrows))

# Achata o array 'axes' para facilitar a iteracao
axes = axes.flatten()

for i, col in enumerate(df_numericas_cols):
  ax = sns.histplot(data=df, x=col, hue=alvo, kde=True, ax=axes[i], palette=paleta, multiple='stack', shrink=0.9)

  axes[i].set_xlabel('')
  axes[i].set_ylabel('Contagem de Pacientes')
  axes[i].set_title(f'Distribuição de {rotulos_vars.get(col, col)}', fontsize=14, fontweight='bold')

  # Adiciona rotulos com as quantidades nas barras
  for container in ax.containers:
    labels = [f'{int(v.get_height())}' if v.get_height() > 0 else '' for v in container]
    ax.bar_label(container, labels=labels, fontsize=9, padding=3)

  # Altera o titulo e os rotulos da legenda diretamente
  legenda = ax.get_legend()
  legenda.set_title('')
  # Atualiza os rotulos da legenda
  if len(legenda.texts) == len(alvo_rotulos):
    for txt_obj, rotulo in zip(legenda.texts, alvo_rotulos):
      txt_obj.set_text(rotulo)

# Remove subplots vazios, se houver
for j in range(num_plots, nrows * ncols_3):
  fig.delaxes(axes[j])

plt.tight_layout() # Ajusta o layout para evitar sobreposicao
plt.show()

In [ ]:
# Converte a coluna de genero para booleano
# 1: `M`, homem | 0: `F`, mulher
if df['Sex'].dtype != 'bool':
  df['Sex'] = df['Sex'].map({'M': 1, 'F': 0}).astype('bool')
print(f"Tipo da coluna Sex: {df['Sex'].dtype}")

# Converte a coluna SES para inteiro mantendo os nulos
df['SES'] = pd.to_numeric(df['SES']).astype('Int64')
print(f"Valores únicos em SES: {df['SES'].unique().tolist()}")

In [ ]:
# Calcula a correlacao entre as variaveis
df_corr = df.corr()
display(df_corr)

# Pega os indexes, ordenando os valores para o menor
id_corr_alvo = df_corr[alvo].abs().sort_values(ascending=False).index

# Calcula a correlacao da coluna alvo com as colunas selecionadas
df_corr_alvo = df[id_corr_alvo].corrwith(df[alvo]).rename(alvo).to_frame()

# Plota o grafico de correlacao (heatmap) com o status de demencia
config_corr_alvo = {
  'vmin': -1,
  'vmax': 1,
}
plotar_heatmap(df_corr_alvo, [4, 4], f'Correlação com {rotulos_vars[alvo]}', **config_corr_alvo)

In [ ]:
# Grafico Pairplot
g = sns.pairplot(df, hue=alvo, diag_kind='hist', palette=paleta, corner=True)

# Ajuste manual da legenda para manter a clareza dos dados
g._legend.set_title(rotulos_vars.get(alvo, alvo))
for i, texto in enumerate(g._legend.texts):
  if i < len(alvo_rotulos):
    texto.set_text(alvo_rotulos[i])

plt.suptitle('Matriz de Dispersão e Distribuição por Status de Demência', fontweight='bold', fontsize=14)
plt.show()

In [ ]:
# Tabela com o perfil de pessoas com e sem demencia, usando mediana e moda
df.groupby(alvo).agg({
  'Sex': obter_moda,
  'Age': 'median',
  'EDUC': 'median',
  'SES': obter_moda,
  'MMSE': 'median',
  'CDR': 'median',
  'eTIV': 'median',
  'nWBV': 'median',
  'ASF': 'median',
})

### Analise inicial

- Inicialmente, tratamos as categorias que deveriam ser booleanas ou categóricas, mas teremos que converter categóricas para numéricas.
- Verificamos que há duas variáveis com valores nulos, e são poucos, então serão tratados após a separação dos dados em treino e teste.
- Verificamos que os valores da variavel alvo são quase iguais, então pode ser que não precisemos ajustá-los com under/over sampling para os modelos.
- A amostra possui mais mulheres que homens e o maior nível socioeconômico é o 2. Não temos acesso ao significado os níveis, mas supomos que quanto maior o número, melhor é o nível do paciente.

## 3. Modelo de ML

Optamos por separar 75% dos dados para treino e 25% para teste.

In [ ]:
def preprocessar_dados(dados, alvo, test_size=0.2, random_state=42):
  """
  Preprocessa o dataframe para modelagem:
  1. Separa variaveis preditoras (X) e alvo (y).
  2. Faz a separacao estratificada dos dados em treino e teste.
  3. Normaliza as variaveis usando StandardScaler (ajustado no treino).
  4. Reconstroi dfs normalizados com colunas e indices originais.

  Parametros:
    dados (pd.DataFrame): df de entrada, ja tratado.
    alvo (str): nome da coluna davariavel alvo.
    test_size (float, opcional): proporcao dos dados para o conjunto de teste. Padrao: 0.2.

  Retorno: tupla: (X_train_scaled, X_test_scaled, y_train, y_test)
  """
  # Separa as variaveis para treino e teste
  X = dados.drop(columns=[alvo])
  y = dados[alvo]

  X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
  )

  # Trata nulos: moda calculada SO no treino, aplicada nos dois
  X_train, X_test = tratar_nulos(X_train, X_test, 'moda')

  # Normaliza com StandardScaler pra nao vazar dado
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)    # fit + transform no treino
  X_test_scaled = scaler.transform(X_test)          # so transform no teste

  # Traz o df de volta com os nomes das colunas e indices originais
  X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
  X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

  return X_train_scaled, X_test_scaled, y_train, y_test


def treinar_prever_modelo(modelo, X_train, y_train, X_test):
  """
  Treina um modelo e gera previsoes e probabilidades.

  Parametros:
    modelo: Instancia do modelo a ser treinado (ex: LogisticRegression(), DecisionTreeClassifier()).
    X_train (pd.DataFrame): Dados de treino (features).
    y_train (pd.Series): Dados de treino (alvo).
    X_test (pd.DataFrame): Dados de teste (features).

  Retorna: Tupla (y_pred, y_prob) com as previsoes e probabilidades do modelo no conjunto de teste.
  """
  modelo.fit(X_train, y_train)
  y_pred = modelo.predict(X_test)
  if hasattr(modelo, 'predict_proba'):
    y_prob = modelo.predict_proba(X_test)[:, 1]
  return y_pred, y_prob


def criar_df_resultados(y_test, y_pred, y_prob):
  """
  Cria um df com os valores reais, previstos e probabilidades para o conjunto de teste.

  Parametros:
    y_test (pd.Series): Valores reais do conjunto de teste.
    y_pred (np.array): Predicos do modelo no conjunto de teste.
    y_prob (np.array): Probabilidades das predicoes do modelo no conjunto de teste.

  Retorna: pd.DataFrame com 'Real', 'Previsto' e 'Probabilidade'.
  """
  df_resultados = pd.DataFrame(index=y_test.index)
  df_resultados['Real'] = y_test
  df_resultados['Previsto'] = y_pred
  if y_prob is not None:
    df_resultados['Probabilidade'] = y_prob
  return df_resultados


def avaliar_modelo(y_test, y_pred, nome_modelo):
  """
  Avalia um modelo de classificacao, calcula metricas e plota a matriz de confusao.

  Parametros:
    y_test (pd.Series): Valores reais do conjunto de teste.
    y_pred (np.array): Predicoes do modelo no conjunto de teste.
    nome_modelo (str): Nome do modelo para o titulo do grafico e DataFrame de metricas.

  Retorna: pd.DataFrame com as metricas de avaliacao do modelo.
  """
  # Matriz de Confusao
  plotar_heatmap(confusion_matrix(y_test, y_pred),
                 titulo=f'Matriz de Confusão \nde {nome_modelo}',
                 tamanho=[4, 4],
                 rotulo_x='Classe Predita',
                 rotulo_y='Classe Real',
                 fmt='g')

  # DataFrame de metricas
  df_metricas = pd.DataFrame({
    'Modelo': [nome_modelo],
    'Acurácia': [accuracy_score(y_test, y_pred)],
    'Precisão': [precision_score(y_test, y_pred)],
    'Recall': [recall_score(y_test, y_pred)],
    'F1-Score': [f1_score(y_test, y_pred)],
  })
  display(df_metricas)
  return df_metricas

In [ ]:
rs = 42 # valor de random_state
# Configuracoes padrao dos modelos
modelo_params = {
  'class_weight': 'balanced',
  'random_state': rs
}
# Cria um novo DataFrame removendo a coluna CDR
df_modelo = df.drop(columns=['CDR'])

X_train_scaled, X_test_scaled, y_train, y_test = preprocessar_dados(df_modelo, alvo, 0.25)

print(f'Tamanho do conjunto de treino: {X_train_scaled.shape[0]} amostras | Nulos: {X_train_scaled.isna().sum().sum()}')
print(f'Tamanho do conjunto de teste: {X_test_scaled.shape[0]} amostras | Nulos: {X_test_scaled.isna().sum().sum()}')

### Regressão Logística

In [ ]:
# Instancia o modelo
logreg = LogisticRegression(max_iter=1000, **modelo_params)
# Treina e testa o modelo
y_pred_lr, y_prob_lr = treinar_prever_modelo(logreg, X_train_scaled, y_train, X_test_scaled)
# Cria um df para os resultados do teste
resultados_logreg = criar_df_resultados(y_test, y_pred_lr, y_prob_lr)
resultados_logreg.head()

In [ ]:
# Plota a matriz de confusao e armazena as metricas do modelo
metricas_lr = avaliar_modelo(y_test, y_pred_lr, 'Regressão Logística')

### Árvore de Decisão

In [ ]:
# Instancia o modelo
dtree = DecisionTreeClassifier(max_depth=5, **modelo_params)
# Treina e testa o modelo
y_pred_dt, y_prob_dt = treinar_prever_modelo(dtree, X_train_scaled, y_train, X_test_scaled)
# Cria um df para os resultados do teste
resultados_dtree = criar_df_resultados(y_test, y_pred_dt, y_prob_dt)
resultados_dtree.head()

In [ ]:
# Plota a matriz de confusao e armazena as metricas do modelo
metricas_dt = avaliar_modelo(y_test, y_pred_dt, 'Árvore de Decisão')

### Random Forest

In [ ]:
# Instancia o modelo
rforest = RandomForestClassifier(n_estimators=100, max_depth=5, **modelo_params)
# Treina e testa o modelo
y_pred_rf, y_prob_rf = treinar_prever_modelo(rforest, X_train_scaled, y_train, X_test_scaled)
# Cria um df para os resultados do teste
resultados_rf = criar_df_resultados(y_test, y_pred_rf, y_prob_rf)
resultados_rf.head()

In [ ]:
# Plota a matriz de confusao e armazena as metricas do modelo
metricas_rf = avaliar_modelo(y_test, y_pred_rf, 'Random Forest')